In [21]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlencode
from dotenv import load_dotenv

# 1.lấy API key từ .env
base_dir = Path.cwd()
if not (base_dir / ".env").exists():
    base_dir = base_dir.parent
load_dotenv(base_dir / ".env")
api_key = os.getenv("OPENALEX_API_KEY")

# 2. nhóm từ khóa tìm kiếm theo lĩnh vực
fields = {
    "Computer Science & AI": "machine learning",
    "Medicine & Healthcare": "clinical medicine",
    "Environmental Science": "climate change",
    "Economics & Business": "business economics",
    "Engineering & Technology": "engineering technology",
    "Psychology & Education": "educational psychology"
}

url = "https://api.openalex.org/works"
headers = {"Authorization": f"Bearer {api_key}"}
papers = []
seen_ids = set()
seen_dois = set()
collected_at = datetime.now(timezone.utc).isoformat()

# 3.lấy tối đa 25 bài duy nhất cho mỗi nhóm
for group, keyword in fields.items():
    count = 0
    page = 1
    while count < 25 and page <= 10:
        params = {
            "search": keyword,
            "filter": "from_publication_date:2015-01-01,to_publication_date:2026-10-11,has_abstract:true",
            "per_page": 100,
            "page": page
        }
        response = requests.get(url, params=params, headers=headers, timeout=45)
        print(group, "- HTTP", response.status_code, "- page", page)
        response.raise_for_status()
        works = response.json().get("results", [])
        if not works:
            break

        for work in works:
            if count == 25:
                break
            paper_id = work.get("id")
            title = work.get("title")
            doi = work.get("doi")
            doi_key = doi.strip().lower() if doi else None
            if not paper_id or not title or paper_id in seen_ids or (doi_key and doi_key in seen_dois):
                continue

            # Tác giả
            authors = []
            for authorship in work.get("authorships", []):
                author = authorship.get("author") or {}
                if author.get("display_name"):
                    authors.append(author["display_name"])

            # Lĩnh vực chính
            topic = work.get("primary_topic") or {}
            field = (topic.get("field") or {}).get("display_name")

            # Tạp chí/hội nghị và nhà xuất bản
            location = work.get("primary_location") or {}
            source = location.get("source") or {}

            # Dựng abstract
            inverted = work.get("abstract_inverted_index") or {}
            words = []
            for word, positions in inverted.items():
                for position in positions:
                    words.append((position, word))
            words.sort()
            abstract = " ".join(word for _, word in words) if words else None

            # Liên kết tìm kiếm Scholar
            scholar_search_url = "https://scholar.google.com/scholar?" + urlencode({"q": f'"{title}"'})

            papers.append({
                "paper_id": paper_id,
                "title": title,
                "query_field": group,
                "field": field,
                "authors": "; ".join(authors) if authors else None,
                "pub_year": work.get("publication_year"),
                "venue": source.get("display_name"),
                "publisher": source.get("host_organization_name"),
                "publication_type": work.get("type"),
                "citations": work.get("cited_by_count"),
                "abstract": abstract,
                "doi": doi,
                "language": work.get("language"),
                "open_access": (work.get("open_access") or {}).get("is_oa"),
                "scholar_search_url": scholar_search_url,
                "collected_at": collected_at,
                "source_database": "OpenAlex",
                "source_url": paper_id
            })
            seen_ids.add(paper_id)
            if doi_key:
                seen_dois.add(doi_key)
            count += 1

        page += 1
        time.sleep(0.3)
    print("Đã thu thập", count, "bài cho nhóm:", group)

# 4. Tạo bảng dữ liệu và lưu data
df = pd.DataFrame(papers)
output_dir = base_dir / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "research_papers_raw.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nKích thước dataset:", df.shape)
print("\nSố bài theo nhóm:")
print(df["query_field"].value_counts())
print("\nSố giá trị thiếu theo cột:")
print(df.isna().sum())
print("\nSố Paper ID bị trùng:", df["paper_id"].duplicated().sum())
print("\nFile CSV:", output_file)
display(df.head())

Computer Science & AI - HTTP 200 - page 1
Đã thu thập 25 bài cho nhóm: Computer Science & AI
Medicine & Healthcare - HTTP 200 - page 1
Đã thu thập 25 bài cho nhóm: Medicine & Healthcare
Environmental Science - HTTP 200 - page 1
Đã thu thập 25 bài cho nhóm: Environmental Science
Economics & Business - HTTP 200 - page 1
Đã thu thập 25 bài cho nhóm: Economics & Business
Engineering & Technology - HTTP 200 - page 1
Đã thu thập 25 bài cho nhóm: Engineering & Technology
Psychology & Education - HTTP 200 - page 1
Đã thu thập 25 bài cho nhóm: Psychology & Education

Kích thước dataset: (150, 18)

Số bài theo nhóm:
query_field
Computer Science & AI       25
Medicine & Healthcare       25
Environmental Science       25
Economics & Business        25
Engineering & Technology    25
Psychology & Education      25
Name: count, dtype: int64

Số giá trị thiếu theo cột:
paper_id               0
title                  0
query_field            0
field                  7
authors               13
pub_year 

,paper_id,title,query_field,field,authors,pub_year,venue,publisher,publication_type,citations,abstract,doi,language,open_access,scholar_search_url,collected_at,source_database,source_url
0,https://openalex.org/W3153990350,Machine learning and deep learning,Computer Science & AI,Computer Science,Christian Janiesch; Patrick Zschech; Kai Heinrich,2021,Electronic Markets,Springer Science+Business Media,article,2655,"Abstract Today, intelligent systems that offer...",https://doi.org/10.1007/s12525-021-00475-2,en,True,https://scholar.google.com/scholar?q=%22Machin...,2026-10-11T07:19:18.803025+00:00,OpenAlex,https://openalex.org/W3153990350
1,https://openalex.org/W3208674021,Machine Learning,Computer Science & AI,Computer Science,Ethem Alpaydın,2021,The MIT Press eBooks,The MIT Press,book,374,A concise overview of machine learning—compute...,https://doi.org/10.7551/mitpress/13811.001.0001,en,True,https://scholar.google.com/scholar?q=%22Machin...,2026-10-11T07:19:18.803025+00:00,OpenAlex,https://openalex.org/W3208674021
2,https://openalex.org/W2751318774,Introduction to Machine Learning,Computer Science & AI,Computer Science,NaN,2019,Series in machine perception and artificial in...,World Scientific,book-chapter,1648,The goal of machine learning is to program com...,https://doi.org/10.1142/9789811201967_0001,en,False,https://scholar.google.com/scholar?q=%22Introd...,2026-10-11T07:19:18.803025+00:00,OpenAlex,https://openalex.org/W2751318774
3,https://openalex.org/W2562618250,Python Machine Learning,Computer Science & AI,Computer Science,Sebastian Raschka,2015,NaN,NaN,book,649,Unlock deeper insights into Machine Leaning wi...,NaN,en,False,https://scholar.google.com/scholar?q=%22Python...,2026-10-11T07:19:18.803025+00:00,OpenAlex,https://openalex.org/W2562618250
4,https://openalex.org/W3010521003,An Introduction to Machine Learning,Computer Science & AI,Computer Science,Solveig Badillo; Balázs Bánfai; Fabian Birzele...,2020,Clinical Pharmacology & Therapeutics,Wiley,article,802,"In the last few years, machine learning (ML) a...",https://doi.org/10.1002/cpt.1796,en,True,https://scholar.google.com/scholar?q=%22An+Int...,2026-10-11T07:19:18.803025+00:00,OpenAlex,https://openalex.org/W3010521003
